# 00: Connection check

Confirms that the kernel can reach the data lake

**Before running**, in a terminal:

```powershell
aws sso login --profile ciccada
```


## Setup

Find the repository root by walking upwards, then put it on `sys.path`, so the
imports work regardless of where Jupyter was started.


In [1]:
# Reload edited .py modules without restarting the kernel.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from bms_sa_review.ami_data_analysis.config import ami_config as C
from bms_sa_review.ami_data_analysis.lib import ami_athena as A
from bms_sa_review.ami_data_analysis.lib import ami_diagnostics as D

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

A.reset_scan_log()

print("Repository root:", REPO_ROOT)
print("Store dir      :", C.STORE_DIR, "(created in Phase 4; absence here is expected)")
print("Databases      :", C.SA, "|", C.SAI, "|", C.BOM_DB)


Repository root: C:\Users\z3553082\OneDrive - UNSW\Documents\GitHub\CICCADA
Store dir      : C:\Users\z3553082\AppData\Local\ciccada\ami_store (created in Phase 4; absence here is expected)
Databases      : solar_analytics | solar_analytics_iceberg | bom_nci


## 1. Environment

In [2]:
env = D.environment_report()
display(env)

missing = env.loc[env["required"] & (env["status"] == "MISSING")]
if len(missing):
    raise RuntimeError(
        "Required packages missing:\n"
        + missing[["item", "detail"]].to_string(index=False)
    )
print("Environment OK.")


,group,item,status,detail,required
0,required,package: pandas,ok,3.0.3,True
1,required,package: numpy,ok,2.4.6,True
2,aws,package: boto3,ok,1.43.40,True
3,aws,package: botocore,ok,1.43.40,True
4,aws,package: awswrangler,ok,3.17.0,True
5,local,package: duckdb,ok,1.5.4,False
6,local,package: pyarrow,ok,24.0.0,False
7,local,package: matplotlib,ok,3.11.0,False
8,local,package: nbformat,ok,5.10.4,False
9,local,package: pytest,ok,9.1.1,False


Environment OK.


## 2. Credentials

Check for expired SSO token

In [4]:
status = A.credential_status()

if status["ok"]:
    print("SSO session valid.")
#    print(f"  profile : {status['profile']}")
#    print(f"  region  : {status['region']}")
#    print(f"  account : {status['account']}")
#    print(f"  identity: {status['arn']}")
else:
    print("SSO SESSION NOT USABLE")
    print(f"  reason : {status['reason']}")
    print(f"  profile: {status['profile']}   region: {status['region']}")
    print()
    print(status["remedy"])

aws_checks = D.aws_report(status)
# display(aws_checks)


SSO session valid.


In [ ]:
# Stop here if the session is dead. Everything below needs it.
# A.require_credentials()

## 3. Both Glue databases are reachable

- `solar_analytics` is the legacy Hive catalogue
- `solar_analytics_iceberg` is the primary one 

In [5]:
cfg = A.get_aws_config()

all_databases = cfg.databases()
display(all_databases)

reachable = {}
for db in (C.SA, C.SAI, C.BOM_DB):
    try:
        listed = cfg.tables(db)
        reachable[db] = len(listed)
        print(f"{db:<28} {len(listed):>4} tables")
    except Exception as exc:
        reachable[db] = None
        print(f"{db:<28} UNREACHABLE -- {type(exc).__name__}: {exc}")


,Database,Description
0,bom_nci,
1,default,Default Hive database
2,elb_logdb,
3,sapn2022,
4,solar_analytics,Migrated from Hive Metastore
5,solar_analytics_iceberg,
6,test_db,
7,type_probe,


solar_analytics                13 tables
solar_analytics_iceberg        35 tables
bom_nci                         1 tables


## 4. Trivial query run


In [6]:
trivial = A.aq("SELECT 1 AS ok", database=C.SAI, label="SELECT 1")
display(trivial)

dim_sample = A.aq("SELECT * FROM circuits LIMIT 5", database=C.SA, label="circuits LIMIT 5")
display(dim_sample)


,ok
0,1


,site_id,device_id,circuit_id,device_type,circuit_polarity,circuit_type,is_pv
0,484720983,135961,84703,Watt Watcher,1,pv_site_net,True
1,484720983,135961,84704,Watt Watcher,1,pv_site_net,True
2,484720983,135961,84705,Watt Watcher,1,pv_site_net,True
3,150652475,140483,658777,Watt Watcher,1,ac_load_net,False
4,150652475,140483,658778,Watt Watcher,1,ac_load_net,False


## 5. The partition guard

`ami_athena.aq()` refuses to run an unpartitioned query against `ts` or the other large tables. 
Nothing here runs a query.


In [7]:
guard_demo = pd.DataFrame([
    {"sql": sql, "blocked": bool(A.check_partition_filters(sql))}
    for sql in [
        "SELECT * FROM ts",
        "SELECT count(*) FROM ts",
        "SELECT * FROM ts WHERE year = 2025 AND month = 1 LIMIT 5",
        'SELECT * FROM "ts$partitions"',
        "SELECT * FROM circuits",
    ]
])
display(guard_demo)

assert guard_demo.blocked.tolist() == [True, True, False, False, False], (
    "The partition guard is not behaving as expected -- do not proceed to 01."
)
print("Partition guard armed.")


,sql,blocked
0,SELECT * FROM ts,True
1,SELECT count(*) FROM ts,True
2,SELECT * FROM ts WHERE year = 2025 AND month =...,False
3,"SELECT * FROM ""ts$partitions""",False
4,SELECT * FROM circuits,False


Partition guard armed.


## 6. Conventions, and what is still unresolved

Every methodological choice this package depends on, with its resolution state.

At Phase 0 almost everything is `resolved = False`. That is the point: these are
declared placeholders awaiting evidence from notebooks 02 and 03, not silent
defaults. `manifest()` will print this same table alongside every result from
Phase 5, so an unresolved choice cannot quietly become a decision.


In [8]:
display(C.describe_conventions())

pending = C.unresolved()
print(f"{len(pending)} convention(s) still unresolved:")
for name in pending:
    print("  -", name)


,convention,value,resolved
0,source dataset,UNRESOLVED,False
1,source interval,5 min,True
2,target AMI interval,30 min,False
3,analysis frame,"AEST (UTC+10, fixed)",True
4,t_stamp frame in `ts`,UTC; partitions are UTC year/month,True
5,power column units,instantaneous W (per Stage 1: /1000 -> kW),False
6,energy_reactive units,5-minute kvarh (per Stage 1: /1000*12 -> avera...,False
7,resample: power,"to interval energy first, then SUM",False
8,resample: energy_reactive,SUM as delivered (do NOT pre-multiply by 12),False
9,circuit_polarity,"applied at extraction, per build_structured_da...",True


12 convention(s) still unresolved:
  - source dataset
  - target AMI interval
  - power column units
  - energy_reactive units
  - resample: power
  - resample: energy_reactive
  - component signs
  - meter signs
  - composition
  - circuit -> signal map
  - aggregate circuit types
  - battery / EV handling


## 7. Verdict


In [9]:
checks = pd.concat([
    aws_checks,
    pd.DataFrame([
        D.check("Glue: solar_analytics reachable", "listed",
                reachable.get(C.SA), reachable.get(C.SA) is not None, group="catalog"),
        D.check("Glue: solar_analytics_iceberg reachable", "listed",
                reachable.get(C.SAI), reachable.get(C.SAI) is not None, group="catalog"),
        D.check("Athena executes", 1,
                None if trivial.empty else int(trivial.ok.iloc[0]),
                (not trivial.empty) and int(trivial.ok.iloc[0]) == 1, group="athena"),
        D.check("dimension read returns rows", "5 rows", f"{len(dim_sample)} rows",
                len(dim_sample) == 5, group="athena"),
    ]),
], ignore_index=True)

display(checks)
assert D.summarise(checks, label="00 connection check"), (
    "Connection check failed -- fix the above before running 01."
)


,group,check,expected,observed,pass,note
0,aws,SSO session valid,valid,valid,True,
1,aws,profile,ciccada,ciccada,True,AWS_PROFILE overrides the default; not an erro...
2,aws,region,ap-southeast-2,ap-southeast-2,True,
3,catalog,Glue: solar_analytics reachable,listed,13,True,
4,catalog,Glue: solar_analytics_iceberg reachable,listed,35,True,
5,athena,Athena executes,1,1,True,
6,athena,dimension read returns rows,5 rows,5 rows,True,


00 connection check: PASS  (7/7 checks)


## 8. Cost check

`source = unavailable` means the scan figure could not be recovered from the
Athena response, **not** that the query was free. The total is a lower bound in
that case.


In [10]:
display(A.scan_report())


2 queries, 1.70 MB scanned, ~AUD 0.0002 (billed at a 10.00 MB minimum per query)


,label,database,n_rows,seconds,scanned,scanned_bytes,cost,source
0,SELECT 1,solar_analytics_iceberg,1,1.92,0 B,0.0,0.0001,query_metadata
1,circuits LIMIT 5,solar_analytics,5,1.84,1.70 MB,1781814.0,0.0001,query_metadata
